## 필수 · 기본 문제 1. 동적 padding과 PAD-mask 일치

### 문제 배경

Dataset에 저장할 때 모든 문장을 최대 길이로 채우면 짧은 batch에도 불필요한 PAD가 많습니다. 먼저 가변 길이로 tokenize하고 collator가 현재 batch의 최장 길이에만 맞추게 합니다.

### 시작 코드

```python
texts = ["금리 전망", "대표팀이 연장전 끝에 결승에 진출했다", "새 반도체 공개"]

def make_dynamic_batch(texts, max_length=16):
    raise NotImplementedError
```

### 수행 요구사항

1. Tokenizer 호출에서 `padding=False`, `truncation=True`를 사용하세요.
2. Encoding별 원래 길이를 기록하세요.
3. `DataCollatorWithPadding(return_tensors="pt")`으로 batch를 만드세요.
4. Batch 두 번째 축이 `min(max(original_lengths), max_length)`인지 확인하세요.
5. PAD ID 위치와 mask 0 위치의 완전 일치를 확인하세요.

### 제출 결과

- 원래 길이 list와 batch shape
- 문장별 PAD 수
- `mask_pad_consistent=True`
- `기본 문제 1 자동 검증: PASS`

### 자동 검증

```python
report = make_dynamic_batch(texts)
assert report["batch_shape"][0] == len(texts)
assert report["batch_shape"][1] == max(report["original_lengths"])
assert report["mask_pad_consistent"] is True
print("기본 문제 1 자동 검증: PASS")
```


    
    **자주 하는 실수**
    
    - Tokenizer에서 이미 `padding="max_length"`를 적용해 동적 padding 효과를 없앱니다.
    - Python list를 `torch.stack`해 길이 불일치 오류를 만듭니다.
    - PAD 수와 실제 token 수를 반대로 해석합니다.

In [8]:
# 최초 1회 환경 확인
import transformers
print("transformers:", transformers.__version__)

transformers: 5.16.1


In [10]:
from transformers import AutoTokenizer

MODEL_ID = "monologg/koelectra-small-v3-discriminator"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

접근 순서 · 입력 list의 최대 길이를 먼저 구한 뒤 PAD ID와 mask 0을 같은 개수만큼 붙입니다. 원본 길이만큼은 mask 1이어야 하므로 두 결과를 함께 만들면 불일치를 줄일 수 있습니다.

In [11]:
from transformers import AutoTokenizer

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
text = "한국어 모델은 문장을 토큰으로 나눕니다."

def inspect_encoding(text, model_id=MODEL_ID):
    # 검수 환경에서는 캐시만 사용해 네트워크 상태와 결과를 분리합니다.
    tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True)
    tokens_without_special = tokenizer.tokenize(text)
    ids_without_special = tokenizer.convert_tokens_to_ids(tokens_without_special)
    # 경계 token을 포함한 실제 model 입력 ID도 별도로 만듭니다.
    ids_with_special = tokenizer.encode(text, add_special_tokens=True)
    tokens_with_special = tokenizer.convert_ids_to_tokens(ids_with_special)
    return {
        "tokenizer_class": type(tokenizer).__name__,
        "vocab_size": tokenizer.vocab_size,
        "special_tokens_map": tokenizer.special_tokens_map,
        "tokens_without_special": tokens_without_special,
        "ids_without_special": ids_without_special,
        "tokens_with_special": tokens_with_special,
        "ids_with_special": ids_with_special,
        "decoded_keep_special": tokenizer.decode(ids_with_special, skip_special_tokens=False),
        "decoded_skip_special": tokenizer.decode(ids_with_special, skip_special_tokens=True),
        "cls_token_id": tokenizer.cls_token_id,
        "sep_token_id": tokenizer.sep_token_id,
    }

result = inspect_encoding(text)
print("class/vocab:", result["tokenizer_class"], result["vocab_size"])
print("tokens:", result["tokens_with_special"])
print("ids:", result["ids_with_special"])
print("decoded:", result["decoded_skip_special"])
assert result["ids_with_special"][0] == result["cls_token_id"]
assert result["ids_with_special"][-1] == result["sep_token_id"]
assert len(result["tokens_without_special"]) + 2 == len(result["ids_with_special"])
print("기본 문제 1 자동 검증: PASS")

class/vocab: BertTokenizer 35000
tokens: ['[CLS]', '한국어', '모델', '##은', '문장', '##을', '토큰', '##으로', '나', '##눕', '##니다', '.', '[SEP]']
ids: [2, 11229, 6918, 4112, 9611, 4292, 32436, 10749, 2236, 4983, 6216, 18, 3]
decoded: 한국어 모델은 문장을 토큰으로 나눕니다.
기본 문제 1 자동 검증: PASS


상세 해설 · Collator는 이 세 문장 중 가장 긴 encoding까지만 PAD합니다. 두 번째 축의 정확한 숫자는 tokenizer 버전과 입력에 따라 달라질 수 있으므로 관계식과 mask 계약을 검증합니다.

## 필수 · 기본 문제 2. BatchEncoding field·shape·PAD 계약

### 문제 배경

길이가 다른 문장을 model에 함께 넣으려면 rectangular Tensor와 mask가 필요합니다. 반환 field를 하드코딩하지 않고 실제 `BatchEncoding`을 검사합니다.

### 시작 코드

```python
texts = ["짧은 문장", "조금 더 긴 한국어 문장을 배치로 처리합니다.", "토큰화 확인"]

def build_and_validate_batch(texts, max_length=16):
    raise NotImplementedError
```

### 수행 요구사항

1. `padding=True`, `truncation=True`, `max_length=16`, `return_tensors="pt"`를 사용하세요.
2. 모든 Tensor field의 shape와 dtype을 수집하세요.
3. `input_ids`와 `attention_mask` shape 일치를 검사하세요.
4. PAD ID 위치와 mask 0 위치가 완전히 같은지 확인하세요.
5. 각 문장의 실제 token 수를 mask 합으로 반환하세요.

### 제출 결과

- Field별 shape/dtype
- Batch shape와 문장별 실제 길이
- `mask_pad_consistent=True`
- `기본 문제 2 자동 검증: PASS`

### 자동 검증

```python
report = build_and_validate_batch(texts)
assert report["batch_shape"][0] == 3 and report["batch_shape"][1] <= 16
assert report["mask_pad_consistent"] is True
assert len(report["true_lengths"]) == 3
print("기본 문제 2 자동 검증: PASS")
```
    
    **자주 하는 실수**
    
    - 모든 모델이 `token_type_ids`를 반환한다고 가정합니다.
    - Mask 합을 PAD 수로 해석합니다. Mask 합은 실제 token 수입니다.
    - `max_length`를 지정하고 `truncation=True`를 빠뜨립니다.

---

검증 순서 · 먼저 실제로 반환된 field 이름과 [B,L] shape를 수집합니다. 그다음 PAD ID가 있는 모든 칸과 attention_mask=0인 칸이 같은지 비교해 조용한 padding 오류를 잡습니다.

In [12]:
import torch
from transformers import AutoTokenizer

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
texts = ["짧은 문장", "조금 더 긴 한국어 문장을 배치로 처리합니다.", "토큰화 확인"]

def build_and_validate_batch(texts, max_length=16):
    # padding=True는 현재 세 문장 중 최장 길이에 맞추는 동적 padding입니다.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    batch = tokenizer(texts, padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt")
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    # PAD가 없는 batch에서도 두 boolean Tensor는 모두 False라 일치합니다.
    # 모든 칸을 비교해 PAD ID와 mask 0이 정확히 같은 격자인지 확인합니다.
    pad_positions = input_ids.eq(tokenizer.pad_token_id)
    zero_mask_positions = attention_mask.eq(0)
    field_contracts = {
        name: {"shape": tuple(value.shape), "dtype": str(value.dtype)}
        for name, value in batch.items()
    }
    return {
        "batch_shape": tuple(input_ids.shape),
        "field_contracts": field_contracts,
        "true_lengths": attention_mask.sum(dim=1).tolist(),
        "mask_pad_consistent": torch.equal(pad_positions, zero_mask_positions),
    }

report = build_and_validate_batch(texts)
print("shape:", report["batch_shape"])
print("fields:", report["field_contracts"])
print("true lengths:", report["true_lengths"])
print("mask_pad_consistent:", report["mask_pad_consistent"])
assert report["batch_shape"][0] == 3 and report["batch_shape"][1] <= 16
assert report["mask_pad_consistent"] is True
assert len(report["true_lengths"]) == 3
print("기본 문제 2 자동 검증: PASS")

shape: (3, 13)
fields: {'input_ids': {'shape': (3, 13), 'dtype': 'torch.int64'}, 'token_type_ids': {'shape': (3, 13), 'dtype': 'torch.int64'}, 'attention_mask': {'shape': (3, 13), 'dtype': 'torch.int64'}}
true lengths: [5, 13, 5]
mask_pad_consistent: True
기본 문제 2 자동 검증: PASS


상세 해설 · padding=True는 이 호출의 최장 문장까지만 채웁니다. 두 번째 축은 텍스트와 tokenizer 버전에 따라 달라질 수 있으므로 <=max_length와 field 간 shape 관계를 검증합니다.